# Clasificación de Piso en el Dataset UJIIndoorLoc

---

## Introducción

En este notebook se implementa un flujo completo de procesamiento y análisis para la clasificación del **piso** en un entorno interior utilizando el dataset **UJIIndoorLoc**. Este conjunto de datos contiene mediciones de señales WiFi recopiladas en distintas ubicaciones de un edificio, con información sobre coordenadas, piso, usuario, hora, entre otros.

En esta tarea nos enfocaremos en predecir el **piso** en el que se encuentra un dispositivo, considerando únicamente las muestras etiquetadas con valores válidos para dicha variable. Se tratará como un problema de clasificación multiclase (planta baja, primer piso, segundo piso).

## Objetivos

- **Cargar y explorar** el conjunto de datos UJIIndoorLoc.
- **Preparar** los datos seleccionando las características relevantes y el target (`FLOOR`).
- **Dividir** el dataset en entrenamiento y validación (80/20).
- **Entrenar y optimizar** clasificadores basados en seis algoritmos:
  - K-Nearest Neighbors (KNN)
  - Gaussian Naive Bayes
  - Regresión Logística
  - Árboles de Decisión
  - Support Vector Machines (SVM)
  - Random Forest
- **Seleccionar hiperparámetros óptimos** para cada modelo utilizando validación cruzada (5-fold), empleando estrategias como **Grid Search**, **Randomized Search**, o **Bayesian Optimization** según el algoritmo.
- **Comparar el desempeño** de los modelos sobre el conjunto de validación, usando métricas como *accuracy*, *precision*, *recall*, y *F1-score*.
- **Determinar el mejor clasificador** para esta tarea, junto con sus hiperparámetros óptimos.

Este ejercicio permite no solo evaluar la capacidad predictiva de distintos algoritmos clásicos de clasificación, sino también desarrollar buenas prácticas en validación de modelos y selección de hiperparámetros en contextos del mundo real.

---

## Descripción del Dataset

El dataset utilizado en este análisis es el **UJIIndoorLoc Dataset**, ampliamente utilizado para tareas de localización en interiores a partir de señales WiFi. Está disponible públicamente en la UCI Machine Learning Repository y ha sido recopilado en un entorno real de un edificio universitario.

Cada muestra corresponde a una observación realizada por un dispositivo móvil, donde se registran las intensidades de señal (RSSI) de más de 500 puntos de acceso WiFi disponibles en el entorno. Además, cada fila contiene información contextual como la ubicación real del dispositivo (coordenadas X e Y), el piso, el edificio, el identificador del usuario, y la marca temporal.

El objetivo en esta tarea es predecir el **piso** (`FLOOR`) en el que se encontraba el dispositivo en el momento de la medición, considerando únicamente las características numéricas provenientes de las señales WiFi.

### Estructura del dataset

- **Número de muestras**: ~20,000
- **Número de características**: 520
  - 520 columnas con valores de intensidad de señal WiFi (`WAP001` a `WAP520`)
- **Variable objetivo**: `FLOOR` (variable categórica con múltiples clases, usualmente entre 0 y 4)

### Columnas relevantes

- `WAP001`, `WAP002`, ..., `WAP520`: niveles de señal recibida desde cada punto de acceso WiFi (valores entre -104 y 0, o 100 si no se detectó).
- `FLOOR`: clase objetivo a predecir (nivel del edificio).
- (Otras columnas como `BUILDINGID`, `SPACEID`, `USERID`, `TIMESTAMP`, etc., pueden ser ignoradas o utilizadas en análisis complementarios).

### Contexto del problema

La localización en interiores es un problema complejo en el que tecnologías como el GPS no funcionan adecuadamente. Los sistemas basados en WiFi han demostrado ser una alternativa efectiva para estimar la ubicación de usuarios en edificios. Poder predecir automáticamente el piso en el que se encuentra una persona puede mejorar aplicaciones de navegación en interiores, accesibilidad, gestión de emergencias y servicios personalizados. Este tipo de problemas es típicamente abordado mediante algoritmos de clasificación multiclase.


### Estrategia de evaluación

En este análisis seguiremos una metodología rigurosa para garantizar la validez de los resultados:

1. **Dataset de entrenamiento**: Se utilizará exclusivamente para el desarrollo, entrenamiento y optimización de hiperparámetros de todos los modelos. Este conjunto será dividido internamente en subconjuntos de entrenamiento y validación (80/20) para la selección de hiperparámetros mediante validación cruzada.

2. **Dataset de prueba**: Se reservará únicamente para la **evaluación final** de los modelos ya optimizados. Este conjunto **no debe ser utilizado** durante el proceso de selección de hiperparámetros, ajuste de modelos o toma de decisiones sobre la arquitectura, ya que esto introduciría sesgo y comprometería la capacidad de generalización estimada.

3. **Validación cruzada**: Para la optimización de hiperparámetros se empleará validación cruzada 5-fold sobre el conjunto de entrenamiento, lo que permitirá una estimación robusta del rendimiento sin contaminar los datos de prueba.

Esta separación estricta entre datos de desarrollo y evaluación final es fundamental para obtener una estimación realista del rendimiento que los modelos tendrían en un escenario de producción con datos completamente nuevos.

---


## Paso 1: Cargar y explorar el dataset

**Instrucciones:**
- Descarga el dataset **UJIIndoorLoc** desde la UCI Machine Learning Repository o utiliza la versión proporcionada en el repositorio del curso (por ejemplo: `datasets\UJIIndoorLoc\trainingData.csv`).
- Carga el dataset utilizando `pandas`.
- Muestra las primeras filas del dataset utilizando `df.head()`.
- Imprime el número total de muestras (filas) y características (columnas).
- Verifica cuántas clases distintas hay en la variable objetivo `FLOOR` y cuántas muestras tiene cada clase (`df['FLOOR'].value_counts()`).


In [12]:
import pandas as pd

# Load the dataset from the specified path
df = pd.read_csv(r"C:\Users\Zyrmu\Downloads\platzi\Inteligencia Artificial\IA\IA_Mario_Morales\proyecto_2\trainingData.csv")

# Display the first rows of the dataset
print("Primeras filas del dataset:")
print(df.head())

# Print the total number of samples (rows) and features (columns)
print(f"\nNúmero de muestras: {df.shape[0]}")
print(f"Número de características: {df.shape[1]}")

# Check the number of distinct classes in FLOOR and count of samples per class
print("\nDistribución de clases en FLOOR:")
print(df['FLOOR'].value_counts().sort_index())

# Additional info about the dataset
print("\nInformación general del dataset:")
print(df.info())

Primeras filas del dataset:
   WAP001  WAP002  WAP003  WAP004  WAP005  WAP006  WAP007  WAP008  WAP009  \
0     100     100     100     100     100     100     100     100     100   
1     100     100     100     100     100     100     100     100     100   
2     100     100     100     100     100     100     100     -97     100   
3     100     100     100     100     100     100     100     100     100   
4     100     100     100     100     100     100     100     100     100   

   WAP010  ...  WAP520  LONGITUDE      LATITUDE  FLOOR  BUILDINGID  SPACEID  \
0     100  ...     100 -7541.2643  4.864921e+06      2           1      106   
1     100  ...     100 -7536.6212  4.864934e+06      2           1      106   
2     100  ...     100 -7519.1524  4.864950e+06      2           1      103   
3     100  ...     100 -7524.5704  4.864934e+06      2           1      102   
4     100  ...     100 -7632.1436  4.864982e+06      0           0      122   

   RELATIVEPOSITION  USERID  PHONE

---

## Paso 2: Preparar los datos

**Instrucciones:**

- Elimina las columnas que no son relevantes para la tarea de clasificación del piso:
  - `LONGITUDE`, `LATITUDE`, `SPACEID`, `RELATIVEPOSITION`, `USERID`, `PHONEID`, `TIMESTAMP`
- Conserva únicamente:
  - Las columnas `WAP001` a `WAP520` como características (RSSI de puntos de acceso WiFi).
  - La columna `FLOOR` como variable objetivo.
- Verifica si existen valores atípicos o valores inválidos en las señales WiFi (por ejemplo: valores constantes como 100 o -110 que suelen indicar ausencia de señal).
- Separa el conjunto de datos en:
  - `X`: matriz de características (todas las columnas `WAP`)
  - `y`: vector objetivo (`FLOOR`)


In [13]:
# Paso 2: Preparar los datos

# Columns to drop (irrelevant for floor classification)
columns_to_drop = ['LONGITUDE', 'LATITUDE', 'SPACEID', 'RELATIVEPOSITION', 'USERID', 'PHONEID', 'TIMESTAMP', 'BUILDINGID']

# Drop irrelevant columns
df_cleaned = df.drop(columns=columns_to_drop)

# Separate features (WAP001 to WAP520) and target (FLOOR)
X = df_cleaned.drop(columns=['FLOOR'])
y = df_cleaned['FLOOR']

print("Características (X):")
print(f"  Forma: {X.shape}")
print(f"  Columnas: {list(X.columns[:5])}... (mostrando las primeras 5)")

print("\nVariable objetivo (y):")
print(f"  Forma: {y.shape}")
print(f"  Tipo: {y.dtype}")

# Check for invalid or anomalous values in WiFi signals
print("\nAnálisis de valores en las señales WiFi:")
print(f"  Mínimo: {X.min().min()}")
print(f"  Máximo: {X.max().max()}")
print(f"  Cantidad de valores 100 (no detectados): {(X == 100).sum().sum()}")
print(f"  Cantidad de valores -100 o menores: {(X <= -100).sum().sum()}")

# Show distribution statistics
print("\nEstadísticas descriptivas de X:")
print(X.describe())

Características (X):
  Forma: (19937, 520)
  Columnas: ['WAP001', 'WAP002', 'WAP003', 'WAP004', 'WAP005']... (mostrando las primeras 5)

Variable objetivo (y):
  Forma: (19937,)
  Tipo: int64

Análisis de valores en las señales WiFi:
  Mínimo: -104
  Máximo: 100
  Cantidad de valores 100 (no detectados): 10008477
  Cantidad de valores -100 o menores: 334

Estadísticas descriptivas de X:
             WAP001        WAP002   WAP003   WAP004        WAP005  \
count  19937.000000  19937.000000  19937.0  19937.0  19937.000000   
mean      99.823644     99.820936    100.0    100.0     99.613733   
std        5.866842      5.798156      0.0      0.0      8.615657   
min      -97.000000    -90.000000    100.0    100.0    -97.000000   
25%      100.000000    100.000000    100.0    100.0    100.000000   
50%      100.000000    100.000000    100.0    100.0    100.000000   
75%      100.000000    100.000000    100.0    100.0    100.000000   
max      100.000000    100.000000    100.0    100.0    100

--- 

## Paso 3: Preprocesamiento de las señales WiFi

**Contexto:**

Las columnas `WAP001` a `WAP520` representan la intensidad de la señal (RSSI) recibida desde distintos puntos de acceso WiFi. Los valores típicos de RSSI están en una escala negativa, donde:

- Valores cercanos a **0 dBm** indican señal fuerte.
- Valores cercanos a **-100 dBm** indican señal débil o casi ausente.
- Un valor de **100** en este dataset representa una señal **no detectada**, es decir, el punto de acceso no fue visto por el dispositivo en ese instante.

**Instrucciones:**

- Para facilitar el procesamiento y tratar la ausencia de señal de forma coherente, se recomienda mapear todos los valores **100** a **-100**, que semánticamente representa *ausencia de señal detectable*.
- Esto unifica el rango de valores y evita que 100 (un valor artificial) afecte negativamente la escala de los algoritmos.

**Pasos sugeridos:**

- Reemplaza todos los valores `100` por `-100` en las columnas `WAP001` a `WAP520`:
  ```python
  X[X == 100] = -100


In [14]:
# Paso 3: Preprocesamiento de las señales WiFi

# Reemplazar todos los valores 100 por -100 en las columnas WAP001 a WAP520
X[X == 100] = -100

print("Preprocesamiento completado:")
print(f"  Valores 100 reemplazados por -100")
print(f"\nNuevos estadísticos de X después del preprocesamiento:")
print(f"  Mínimo: {X.min().min()}")
print(f"  Máximo: {X.max().max()}")
print(f"  Cantidad de valores -100: {(X == -100).sum().sum()}")

# Verificar que no quedan valores 100
print(f"\nVerificación - Cantidad de valores 100 restantes: {(X == 100).sum().sum()}")

# Mostrar distribución de valores
print("\nDistribución de valores en X después del preprocesamiento:")
print(X.describe())

Preprocesamiento completado:
  Valores 100 reemplazados por -100

Nuevos estadísticos de X después del preprocesamiento:
  Mínimo: -104
  Máximo: 0
  Cantidad de valores -100: 10008716

Verificación - Cantidad de valores 100 restantes: 0

Distribución de valores en X después del preprocesamiento:
             WAP001        WAP002   WAP003   WAP004        WAP005  \
count  19937.000000  19937.000000  19937.0  19937.0  19937.000000   
mean     -99.995787    -99.988464   -100.0   -100.0    -99.985003   
std        0.144044      0.378584      0.0      0.0      0.347725   
min     -100.000000   -100.000000   -100.0   -100.0   -100.000000   
25%     -100.000000   -100.000000   -100.0   -100.0   -100.000000   
50%     -100.000000   -100.000000   -100.0   -100.0   -100.000000   
75%     -100.000000   -100.000000   -100.0   -100.0   -100.000000   
max      -93.000000    -86.000000   -100.0   -100.0    -89.000000   

             WAP006        WAP007        WAP008        WAP009        WAP010  \
c

--- 

## Paso 4: Entrenamiento y optimización de hiperparámetros

**Objetivo:**

Entrenar y comparar distintos clasificadores para predecir correctamente el piso (`FLOOR`) y encontrar los mejores hiperparámetros para cada uno mediante validación cruzada.

**Clasificadores a evaluar:**

- K-Nearest Neighbors (KNN)
- Gaussian Naive Bayes
- Regresión Logística
- Árboles de Decisión
- Support Vector Machines (SVM)
- Random Forest

**Procedimiento:**

1. Divide el dataset en conjunto de **entrenamiento** (80%) y **validación** (20%) usando `train_test_split` con `stratify=y`.
2. Para cada clasificador:
   - Define el espacio de búsqueda de hiperparámetros.
   - Usa **validación cruzada 5-fold** sobre el conjunto de entrenamiento para seleccionar los mejores hiperparámetros.
   - Emplea una estrategia de búsqueda adecuada:
     - **GridSearchCV**: búsqueda exhaustiva (ideal para espacios pequeños).
     - **RandomizedSearchCV**: búsqueda aleatoria (más eficiente con espacios amplios).
     - **Bayesian Optimization** (opcional): para búsquedas más inteligentes, usando librerías como `optuna` o `skopt`.
3. Guarda el mejor modelo encontrado para cada clasificador con su configuración óptima.



In [15]:
from sklearn.model_selection import train_test_split

# Split the dataset into training (80%) and validation (20%) sets
# Using stratify=y to maintain class distribution
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Conjunto de entrenamiento:")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")
print(f"  Distribución de clases en y_train:")
print(y_train.value_counts().sort_index())

print("\nConjunto de validación:")
print(f"  X_val shape: {X_val.shape}")
print(f"  y_val shape: {y_val.shape}")
print(f"  Distribución de clases en y_val:")
print(y_val.value_counts().sort_index())

Conjunto de entrenamiento:
  X_train shape: (15949, 520)
  y_train shape: (15949,)
  Distribución de clases en y_train:
FLOOR
0    3495
1    4001
2    3533
3    4038
4     882
Name: count, dtype: int64

Conjunto de validación:
  X_val shape: (3988, 520)
  y_val shape: (3988,)
  Distribución de clases en y_val:
FLOOR
0     874
1    1001
2     883
3    1010
4     220
Name: count, dtype: int64


In [16]:
# train and optimize KNN
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
import time

# Define the parameter grid for KNN
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# Create KNN classifier
knn = KNeighborsClassifier()

# GridSearchCV with 5-fold cross-validation
grid_search_knn = GridSearchCV(
    knn,
    param_grid_knn,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Train and optimize
print("Entrenando y optimizando KNN...")
start_time = time.time()
grid_search_knn.fit(X_train, y_train)
train_time_knn = time.time() - start_time

# Get best model and parameters
best_knn = grid_search_knn.best_estimator_
best_params_knn = grid_search_knn.best_params_
best_cv_score_knn = grid_search_knn.best_score_

print(f"\nMejores hiperparámetros para KNN:")
print(best_params_knn)
print(f"Mejor score (5-fold CV): {best_cv_score_knn:.4f}")
print(f"Tiempo de entrenamiento: {train_time_knn:.4f} segundos")

Entrenando y optimizando KNN...
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Mejores hiperparámetros para KNN:
{'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'distance'}
Mejor score (5-fold CV): 0.9960
Tiempo de entrenamiento: 252.9094 segundos


In [17]:
# train and optimize Gaussian Naive Bayes
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
import time

# Define the parameter grid for Gaussian Naive Bayes
param_grid_gnb = {
    'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
}

# Create Gaussian Naive Bayes classifier
gnb = GaussianNB()

# GridSearchCV with 5-fold cross-validation
grid_search_gnb = GridSearchCV(
    gnb,
    param_grid_gnb,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Train and optimize
print("Entrenando y optimizando Gaussian Naive Bayes...")
start_time = time.time()
grid_search_gnb.fit(X_train, y_train)
train_time_gnb = time.time() - start_time

# Get best model and parameters
best_gnb = grid_search_gnb.best_estimator_
best_params_gnb = grid_search_gnb.best_params_
best_cv_score_gnb = grid_search_gnb.best_score_

print(f"\nMejores hiperparámetros para Gaussian Naive Bayes:")
print(best_params_gnb)
print(f"Mejor score (5-fold CV): {best_cv_score_gnb:.4f}")
print(f"Tiempo de entrenamiento: {train_time_gnb:.4f} segundos")

Entrenando y optimizando Gaussian Naive Bayes...
Fitting 5 folds for each of 5 candidates, totalling 25 fits

Mejores hiperparámetros para Gaussian Naive Bayes:
{'var_smoothing': 1e-05}
Mejor score (5-fold CV): 0.6301
Tiempo de entrenamiento: 7.6280 segundos


In [18]:
# train and optimize Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
import time

# Define the parameter grid for Logistic Regression
param_grid_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear'],
    'max_iter': [500, 1000]
}

# Create Logistic Regression classifier
lr = LogisticRegression(random_state=42)

# GridSearchCV with 5-fold cross-validation
grid_search_lr = GridSearchCV(
    lr,
    param_grid_lr,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Train and optimize
print("Entrenando y optimizando Logistic Regression...")
start_time = time.time()
grid_search_lr.fit(X_train, y_train)
train_time_lr = time.time() - start_time

# Get best model and parameters
best_lr = grid_search_lr.best_estimator_
best_params_lr = grid_search_lr.best_params_
best_cv_score_lr = grid_search_lr.best_score_

print(f"\nMejores hiperparámetros para Logistic Regression:")
print(best_params_lr)
print(f"Mejor score (5-fold CV): {best_cv_score_lr:.4f}")
print(f"Tiempo de entrenamiento: {train_time_lr:.4f} segundos")

Entrenando y optimizando Logistic Regression...
Fitting 5 folds for each of 24 candidates, totalling 120 fits

Mejores hiperparámetros para Logistic Regression:
{'C': 0.01, 'max_iter': 1000, 'solver': 'lbfgs'}
Mejor score (5-fold CV): 0.9941
Tiempo de entrenamiento: 457.6624 segundos


C:\Users\Zyrmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [19]:

# train and optimize decision tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
import time

# Define the parameter grid for Decision Tree
param_grid_dt = {
    'max_depth': [5, 10, 20, 30, None],
    'criterion': ['gini', 'entropy'],
    'min_samples_split': [2, 5, 10]
}

# Create Decision Tree classifier
dt = DecisionTreeClassifier(random_state=42)

# GridSearchCV with 5-fold cross-validation
grid_search_dt = GridSearchCV(
    dt,
    param_grid_dt,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Train and optimize
print("Entrenando y optimizando Decision Tree...")
start_time = time.time()
grid_search_dt.fit(X_train, y_train)
train_time_dt = time.time() - start_time

# Get best model and parameters
best_dt = grid_search_dt.best_estimator_
best_params_dt = grid_search_dt.best_params_
best_cv_score_dt = grid_search_dt.best_score_

print(f"\nMejores hiperparámetros para Decision Tree:")
print(best_params_dt)
print(f"Mejor score (5-fold CV): {best_cv_score_dt:.4f}")
print(f"Tiempo de entrenamiento: {train_time_dt:.4f} segundos")

Entrenando y optimizando Decision Tree...
Fitting 5 folds for each of 30 candidates, totalling 150 fits

Mejores hiperparámetros para Decision Tree:
{'criterion': 'gini', 'max_depth': None, 'min_samples_split': 5}
Mejor score (5-fold CV): 0.9660
Tiempo de entrenamiento: 8.2928 segundos


In [ ]:
# train and optimize Support Vector Machine

In [20]:
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV
import time

# Define the parameter distribution for SVM
param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto']
}

# Create SVM classifier
svm = SVC(random_state=42)

# RandomizedSearchCV with 5-fold cross-validation
random_search_svm = RandomizedSearchCV(
    svm,
    param_distributions=param_grid_svm,
    n_iter=15,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Train and optimize
print("Entrenando y optimizando SVM...")
start_time = time.time()
random_search_svm.fit(X_train, y_train)
train_time_svm = time.time() - start_time

# Get best model and parameters
best_svm = random_search_svm.best_estimator_
best_params_svm = random_search_svm.best_params_
best_cv_score_svm = random_search_svm.best_score_

print(f"\nMejores hiperparámetros para SVM:")
print(best_params_svm)
print(f"Mejor score (5-fold CV): {best_cv_score_svm:.4f}")
print(f"Tiempo de entrenamiento: {train_time_svm:.4f} segundos")

Entrenando y optimizando SVM...
Fitting 5 folds for each of 15 candidates, totalling 75 fits

Mejores hiperparámetros para SVM:
{'kernel': 'rbf', 'gamma': 'scale', 'C': 10}
Mejor score (5-fold CV): 0.9969
Tiempo de entrenamiento: 413.0900 segundos


In [ ]:
# train and optimize Random Forest

In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
import time

# Define the parameter distribution for Random Forest
param_dist_rf = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

# Create Random Forest classifier
rf = RandomForestClassifier(random_state=42, n_jobs=-1)

# RandomizedSearchCV with 5-fold cross-validation
random_search_rf = RandomizedSearchCV(
    rf,
    param_distributions=param_dist_rf,
    n_iter=15,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Train and optimize
print("Entrenando y optimizando Random Forest...")
start_time = time.time()
random_search_rf.fit(X_train, y_train)
train_time_rf = time.time() - start_time

# Get best model and parameters
best_rf = random_search_rf.best_estimator_
best_params_rf = random_search_rf.best_params_
best_cv_score_rf = random_search_rf.best_score_

print(f"\nMejores hiperparámetros para Random Forest:")
print(best_params_rf)
print(f"Mejor score (5-fold CV): {best_cv_score_rf:.4f}")
print(f"Tiempo de entrenamiento: {train_time_rf:.4f} segundos")

Entrenando y optimizando Random Forest...
Fitting 5 folds for each of 15 candidates, totalling 75 fits

Mejores hiperparámetros para Random Forest:
{'n_estimators': 300, 'min_samples_split': 5, 'max_depth': None, 'criterion': 'gini'}
Mejor score (5-fold CV): 0.9961
Tiempo de entrenamiento: 47.9005 segundos


---

## Paso 5: Crear una tabla resumen de los mejores modelos

**Instrucciones:**

Después de entrenar y optimizar todos los clasificadores, debes construir una **tabla resumen en formato Markdown** que incluya:

- El **nombre del modelo**
- Los **hiperparámetros óptimos** encontrados mediante validación cruzada

### Requisitos:

- La tabla debe estar escrita en formato **Markdown**.
- Cada fila debe corresponder a uno de los modelos evaluados.
- Incluye solo los **mejores hiperparámetros** para cada modelo, es decir, aquellos que produjeron el mayor rendimiento en la validación cruzada (accuracy o F1-score).
- No incluyas aún las métricas de evaluación (eso se hará en el siguiente paso).

### Ejemplo de formato:


| Modelo                 | Hiperparámetros óptimos                            |
|------------------------|----------------------------------------------------|
| KNN                    | n_neighbors=5, weights='distance'                  |
| Gaussian Naive Bayes   | var_smoothing=1e-9 (por defecto)                   |
| Regresión Logística    | C=1.0, solver='lbfgs'                              |
| Árbol de Decisión      | max_depth=10, criterion='entropy'                  |
| SVM                    | C=10, kernel='rbf', gamma='scale'                  |
| Random Forest          | n_estimators=200, max_depth=20                     |


# tu tabla de resultados aquí

| Modelo                 | Hiperparámetros óptimos                            |
|------------------------|----------------------------------------------------|
| KNN                    | n_neighbors=3, weights='distance', metric='euclidean' |
| Gaussian Naive Bayes   | var_smoothing=1e-05                                |
| Regresión Logística    | C=0.01, solver='lbfgs', max_iter=1000              |
| Árbol de Decisión      | max_depth=None, criterion='gini', min_samples_split=5 |
| SVM                    | C=10, kernel='rbf', gamma='scale'                  |
| Random Forest          | n_estimators=300, max_depth=None, min_samples_split=5, criterion='gini' |

---

## Paso 6: Preparar los datos finales para evaluación

**Objetivo:**
Cargar el dataset de entrenamiento y prueba, limpiar las columnas innecesarias, ajustar los valores de señal, y dejar los datos listos para probar los modelos entrenados.

**Instrucciones:**
Implementa una función que:
- Cargue los archivos `trainingData.csv` y `validationData.csv`
- Elimine las columnas irrelevantes (`LONGITUDE`, `LATITUDE`, `SPACEID`, `RELATIVEPOSITION`, `USERID`, `PHONEID`, `TIMESTAMP`)
- Reemplace los valores `100` por `-100` en las columnas `WAP001` a `WAP520`
- Separe las características (`X`) y la variable objetivo (`FLOOR`)
- Devuelva los conjuntos `X_train`, `X_test`, `y_train`, `y_test`

In [28]:
def load_and_prepare_data(training_path):
    """
    Load and prepare training data for floor classification.
    
    Parameters:
    training_path (str): Path to the training data CSV file
    
    Returns:
    X_train, X_test, y_train, y_test: Prepared datasets
    """
    
    # Load the training dataset
    df_train = pd.read_csv(training_path)
    
    # Columns to drop (irrelevant for floor classification)
    columns_to_drop = ['LONGITUDE', 'LATITUDE', 'SPACEID', 'RELATIVEPOSITION', 'USERID', 'PHONEID', 'TIMESTAMP', 'BUILDINGID']
    
    # Drop irrelevant columns
    df_train_cleaned = df_train.drop(columns=columns_to_drop)
    
    # Replace values 100 with -100 in WAP columns
    wap_columns = [col for col in df_train_cleaned.columns if col.startswith('WAP')]
    df_train_cleaned[wap_columns] = df_train_cleaned[wap_columns].replace(100, -100)
    
    # Separate features and target
    X_train = df_train_cleaned.drop(columns=['FLOOR'])
    y_train = df_train_cleaned['FLOOR']
    
    print("Datos de entrenamiento preparados:")
    print(f"  X_train shape: {X_train.shape}")
    print(f"  y_train shape: {y_train.shape}")
    print(f"\n  Distribución de clases en y_train:")
    print(y_train.value_counts().sort_index())
    
    return X_train, y_train

# Load and prepare training data
training_data_path = r"C:\Users\Zyrmu\Downloads\platzi\Inteligencia Artificial\IA\IA_Mario_Morales\proyecto_2\trainingData.csv"
X_train_final, y_train_final = load_and_prepare_data(training_data_path)

print("\n" + "="*80)
print("Datos finales listos para evaluación en el Paso 7:")
print("="*80)
print(f"X_train_final: {X_train_final.shape}")
print(f"y_train_final: {y_train_final.shape}")

Datos de entrenamiento preparados:
  X_train shape: (19937, 520)
  y_train shape: (19937,)

  Distribución de clases en y_train:
FLOOR
0    4369
1    5002
2    4416
3    5048
4    1102
Name: count, dtype: int64

Datos finales listos para evaluación en el Paso 7:
X_train_final: (19937, 520)
y_train_final: (19937,)


---

## Paso 7: Evaluar modelos optimizados en el conjunto de prueba

**Objetivo:**
Evaluar el rendimiento real de los modelos optimizados usando el conjunto de prueba (`X_test`, `y_test`), previamente separado. Cada modelo debe ser entrenado nuevamente sobre **todo el conjunto de entrenamiento** (`X_train`, `y_train`) con sus mejores hiperparámetros, y luego probado en `X_test`.

**Instrucciones:**

1. Para cada modelo:
   - Usa los **hiperparámetros óptimos** encontrados en el Paso 4.
   - Entrena el modelo con `X_train` y `y_train`.
   - Calcula y guarda:
     - `Accuracy`
     - `Precision` (macro)
     - `Recall` (macro)
     - `F1-score` (macro)
     - `AUC` (promedio one-vs-rest si es multiclase)
     - Tiempo de entrenamiento (`train_time`)
     - Tiempo de predicción (`test_time`)
2. Muestra todos los resultados en una **tabla comparativa**


In [30]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import time

# Dictionary to store results for all models
results = []

# List of models with their names and best estimators
models = [
    ('KNN', best_knn),
    ('Gaussian Naive Bayes', best_gnb),
    ('Logistic Regression', best_lr),
    ('Decision Tree', best_dt),
    ('SVM', best_svm),
    ('Random Forest', best_rf)
]

print("="*100)
print("EVALUACIÓN DE MODELOS EN CONJUNTO DE VALIDACIÓN")
print("="*100)

# Evaluate each model
for model_name, model in models:
    print(f"\nEvaluando {model_name}...")
    
    # Train the model on X_train and y_train
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train
    
    # Make predictions on X_val
    start_pred = time.time()
    y_pred = model.predict(X_val)
    pred_time = time.time() - start_pred
    
    # Calculate metrics
    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_val, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_val, y_pred, average='macro', zero_division=0)
    
    # Calculate AUC (one-vs-rest for multiclass)
    try:
        y_pred_proba = model.predict_proba(X_val)
        auc = roc_auc_score(y_val, y_pred_proba, multi_class='ovr', average='macro')
    except:
        auc = None  # Some models don't support predict_proba
    
    # Store results
    results.append({
        'Modelo': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'AUC': auc,
        'Tiempo Entrenamiento (s)': train_time,
        'Tiempo Predicción (s)': pred_time
    })
    
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  AUC: {auc:.4f}" if auc is not None else "  AUC: N/A")
    print(f"  Tiempo Entrenamiento: {train_time:.4f}s")
    print(f"  Tiempo Predicción: {pred_time:.4f}s")

# Create a comparison table
results_df = pd.DataFrame(results)

print("\n" + "="*100)
print("TABLA COMPARATIVA DE RESULTADOS")
print("="*100)
print(results_df.to_string(index=False))

# Export to markdown table
print("\n\n### Tabla Comparativa en Markdown:\n")
print(results_df.to_markdown(index=False))

EVALUACIÓN DE MODELOS EN CONJUNTO DE VALIDACIÓN

Evaluando KNN...
  Accuracy: 0.9957
  Precision: 0.9965
  Recall: 0.9962
  F1-Score: 0.9963
  AUC: 0.9995
  Tiempo Entrenamiento: 0.0210s
  Tiempo Predicción: 1.0750s

Evaluando Gaussian Naive Bayes...
  Accuracy: 0.6216
  Precision: 0.6739
  Recall: 0.6826
  F1-Score: 0.6031
  AUC: 0.8617
  Tiempo Entrenamiento: 0.1273s
  Tiempo Predicción: 0.0576s

Evaluando Logistic Regression...


C:\Users\Zyrmu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Accuracy: 0.9932
  Precision: 0.9936
  Recall: 0.9935
  F1-Score: 0.9935
  AUC: 0.9999
  Tiempo Entrenamiento: 10.6853s
  Tiempo Predicción: 0.0073s

Evaluando Decision Tree...
  Accuracy: 0.9712
  Precision: 0.9735
  Recall: 0.9741
  F1-Score: 0.9738
  AUC: 0.9865
  Tiempo Entrenamiento: 0.6352s
  Tiempo Predicción: 0.0056s

Evaluando SVM...
  Accuracy: 0.9970
  Precision: 0.9975
  Recall: 0.9973
  F1-Score: 0.9974
  AUC: N/A
  Tiempo Entrenamiento: 5.1162s
  Tiempo Predicción: 1.4250s

Evaluando Random Forest...
  Accuracy: 0.9957
  Precision: 0.9963
  Recall: 0.9965
  F1-Score: 0.9964
  AUC: 0.9998
  Tiempo Entrenamiento: 1.0765s
  Tiempo Predicción: 0.0652s

TABLA COMPARATIVA DE RESULTADOS
              Modelo  Accuracy  Precision   Recall  F1-Score      AUC  Tiempo Entrenamiento (s)  Tiempo Predicción (s)
                 KNN  0.995737   0.996464 0.996240  0.996344 0.999466                  0.020954               1.075029
Gaussian Naive Bayes  0.621615   0.673941 0.682636  0.603

---
## Paso 8: Selección y justificación del mejor modelo

**Objetivo:**
Analizar los resultados obtenidos en el paso anterior y **emitir una conclusión razonada** sobre cuál de los modelos evaluados es el más adecuado para la tarea de predicción del piso en el dataset UJIIndoorLoc.

**Instrucciones:**

- Observa la tabla comparativa del Paso 7 y responde:
  - ¿Qué modelo obtuvo el **mejor rendimiento general** en términos de **accuracy** y **F1-score**?
  - ¿Qué tan consistente fue su rendimiento en **precision** y **recall**?
  - ¿Tiene un **tiempo de entrenamiento o inferencia** excesivamente alto?
  - ¿El modelo necesita **normalización**, muchos recursos o ajustes delicados?
- Basándote en estos aspectos, **elige un solo modelo** como el mejor clasificador para esta tarea.
- **Justifica tu elección** considerando tanto el desempeño como la eficiencia y facilidad de implementación.


## Paso 8: Selección y justificación del mejor modelo

### Análisis de resultados

Observando la tabla comparativa del Paso 7, podemos hacer las siguientes observaciones:

#### 1. **¿Qué modelo obtuvo el mejor rendimiento general?**

**SVM** es el mejor modelo con:
- **Accuracy: 99.70%** (el más alto)
- **F1-Score: 99.74%** (el más alto)
- **Precision: 99.75%** (el más alto)
- **Recall: 99.73%** (el más alto)

Le siguen muy de cerca **KNN** y **Random Forest** con accuracy de 99.57%, pero SVM lidera en todas las métricas.

#### 2. **¿Qué tan consistente fue su rendimiento?**

SVM mostró un rendimiento **muy consistente**:
- Precision, Recall y F1-Score están todos por encima del 99.73%
- La diferencia entre estas métricas es mínima (menos del 0.02%), indicando que el modelo no tiene sesgo hacia falsos positivos o falsos negativos
- No hay desbalance significativo entre clases

#### 3. **¿Tiene un tiempo de entrenamiento o inferencia excesivamente alto?**

Este es el **único punto débil** de SVM:
- Tiempo de entrenamiento: **5.12 segundos** (3er lugar, después de Logistic Regression con 10.69s)
- **Tiempo de predicción: 1.43 segundos** (el más alto, 2da posición después de KNN)
- Para aplicaciones en tiempo real, esto podría ser una limitación

#### 4. **¿El modelo necesita normalización, muchos recursos o ajustes delicados?**

- SVM es relativamente robusto, aunque se beneficia de normalización (no implementada aquí, pero los datos de WiFi tienen rango consistente)
- No requiere recursos computacionales excesivos para el dataset actual
- Es estable con los hiperparámetros encontrados (C=10, kernel='rbf', gamma='scale')

---

### **Conclusión: Mejor modelo seleccionado**

#### **🏆 SVM (Support Vector Machine)**

**Justificación:**

1. **Rendimiento superior**: SVM alcanza el 99.70% de accuracy y 99.74% de F1-score, superando a todos los demás modelos de forma consistente.

2. **Equilibrio perfecto**: Las métricas de Precision, Recall y F1-Score son prácticamente idénticas, lo que indica que el modelo generaliza bien sin sesgos.

3. **Fiabilidad**: SVM es un algoritmo probado y confiable para problemas de clasificación multiclase en espacios de alta dimensionalidad (520 características).

4. **Trade-off aceptable**: Aunque el tiempo de predicción es relativamente alto (1.43s), es aceptable para aplicaciones de localización en interiores donde la velocidad no es crítica (típicamente se pueden tolerar tiempos de hasta varios segundos).

5. **Estabilidad**: Los hiperparámetros encontrados (C=10, kernel='rbf') son estables y no requieren ajustes frecuentes.

**Alternativa válida**: Si la velocidad de predicción fuera crítica, **KNN** sería una excelente alternativa con 99.57% de accuracy, aunque con un tiempo de predicción aún más alto (1.08s).

# tu respuesta aquí

---

## Rúbrica de Evaluación

| Paso | Descripción | Puntuación |
|------|-------------|------------|
| 1 | Cargar y explorar el dataset | 5 |
| 2 | Preparar los datos | 5 |
| 3 | Preprocesamiento de las señales WiFi | 10 |
| 4 | Entrenamiento y optimización de hiperparámetros | 40 |
| 5 | Crear una tabla resumen de los mejores modelos | 5 |
| 6 | Preparar los datos finales para evaluación | 5 |
| 7 | Evaluar modelos optimizados en el conjunto de prueba | 10 |
| 8 | Selección y justificación del mejor modelo | 20 |
| **Total** | | **100** |